In [ ]:
import os
import sys

from datasets import Image, concatenate_datasets, load_dataset

from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTxt

# 1. Configuración
HF_DATASET_ID = "JJMack/pokemon-classification-gen1-9"
URL_ORIGEN_HUGGINGFACE = f"https://huggingface.co/datasets/{HF_DATASET_ID}"

archivo_csv_orig = "dataset_pokemon.csv"
archivo_json_orig = "dataset_pokemon.json"

archivo_csv_nuevo = "dataset_pokemon_modificado.csv"
archivo_json_nuevo = "dataset_pokemon_modificado.json"


def preparar_archivos_locales(ruta_csv, ruta_json):
    if os.path.exists(ruta_csv) and os.path.exists(ruta_json):
        print("-> Los archivos locales ya existen, se reutilizan.\n")
        return

    print("Descargando dataset desde Hugging Face...")
    ds_dict = load_dataset(HF_DATASET_ID)  # splits: train, validation, test

    # Unir los tres splits en una sola tabla
    ds = concatenate_datasets([ds_dict[split] for split in ds_dict])

    # Quitar columnas de imagen para quedarnos solo con los metadatos
    columnas_imagen = [col for col, feat in ds.features.items() if isinstance(feat, Image)]
    ds = ds.remove_columns(columnas_imagen)
    print(f"-> Columnas conservadas: {ds.column_names}")
    print(f"-> Total de registros: {len(ds)}")

    # Pasar a pandas: Dataset.to_json() no genera un JSON válido (lista) con lines=False
    # cuando el batch_size es menor que el número de filas.
    df = ds.to_pandas()

    # Exportar a CSV y a JSON (lista de objetos)
    df.to_csv(ruta_csv, index=False, encoding="utf-8")
    print(f"-> CSV generado en: {ruta_csv}")
    df.to_json(ruta_json, orient="records", force_ascii=False, indent=4)
    print(f"-> JSON generado en: {ruta_json}\n")


try:
    preparar_archivos_locales(archivo_csv_orig, archivo_json_orig)
except Exception as e:
    print(f"Error al preparar el dataset: {e}")
    sys.exit(1)

# Inicializar el generador de resumen con la URL oficial del dataset
resumen_generator = GeneradorResumenTxt(
    url_csv=URL_ORIGEN_HUGGINGFACE,
    url_json=URL_ORIGEN_HUGGINGFACE,
)

# 2. Ejecutar y procesar CSV
print("--- Ejecutando Manejador CSV ---")
csv_handler = ManejadorCSV(archivo_csv_orig)
_, filas_csv = csv_handler.leer_datos()
csv_handler.modificar_y_guardar(archivo_csv_nuevo)

resumen_generator.registrar_proceso(
    formato="CSV",
    archivo_orig=archivo_csv_orig,
    archivo_nuevo=archivo_csv_nuevo,
    modificacion="Actualización del identificador en la primera celda del dataset de Pokémon.",
    registros=len(filas_csv),
    clase="ManejadorCSV",
    metodos=["leer_datos", "modificar_y_guardar"],
)

# 3. Ejecutar y procesar JSON
print("\n--- Ejecutando Manejador JSON ---")
json_handler = ManejadorJSON(archivo_json_orig)
datos_json = json_handler.leer_datos()
json_handler.modificar_y_guardar(archivo_json_nuevo)

# Cantidad de registros (soporta lista o diccionario)
num_registros_json = len(datos_json) if isinstance(datos_json, list) else 1

resumen_generator.registrar_proceso(
    formato="JSON",
    archivo_orig=archivo_json_orig,
    archivo_nuevo=archivo_json_nuevo,
    modificacion="Adición del campo de estado 'estado_revision': 'verificado_gen1_9'.",
    registros=num_registros_json,
    clase="ManejadorJSON",
    metodos=["leer_datos", "modificar_y_guardar"],
)

# 4. Generar el archivo de texto final
archivo_txt_final = resumen_generator.guardar_resumen("resumen_ejecucion.txt")

-> Los archivos locales ya existen, se reutilizan.

--- Ejecutando Manejador CSV ---
Archivo CSV modificado guardado exitosamente en: 'dataset_pokemon_modificado.csv'

--- Ejecutando Manejador JSON ---
Archivo JSON modificado guardado exitosamente en: 'dataset_pokemon_modificado.json'
Resumen consolidado generado exitosamente en: 'resumen_ejecucion.txt'
